[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_44_Document_AI.ipynb)

# Lesson 44 — Track 4 · Document AI: PDFs, Tables & Forms as Agent Tools

**Track 4 Roadmap:**

| Lesson | Topic | Status |
|--------|-------|--------|
| L42 | Voice Agent Pipelines (ASR → LLM → TTS) | ✅ Done |
| L43 | Image Generation as an Agent Tool | ✅ Done |
| **L44** | **Document AI: PDFs, Tables & Forms** | **← You are here** |
| L45 | Track 4 Capstone — Multimodal Agent | ⏳ Next |

## What You'll Build Today

A **Document AI Agent** that can:
- Read and understand PDF documents (invoices, research papers, contracts)
- Extract structured data (tables, forms, key fields)
- Combine text extraction + Claude vision for any document type
- Wire document tools into an agent loop

## Why This Matters

Documents are **everywhere** in enterprise AI:
- 80% of enterprise data is unstructured (PDFs, Word docs, scans)
- Invoice processing, contract review, research extraction are \$100B+ markets
- A document-aware agent closes the "real world data" gap in your agent stack

By the end of this lesson, your agent can ingest a PDF and turn it into structured data ready for action.

In [ ]:
# Setup — run this cell first
!pip install anthropic pdfplumber pypdf fpdf2 pillow tabulate -q

import os
import base64
import json
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Tuple
from pprint import pprint

import anthropic

# Load API key from Colab Secrets (one-time setup: Secrets tab → ANTHROPIC_API_KEY)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set ANTHROPIC_API_KEY in Colab Secrets or as env var")
    print("✅ API key loaded from environment")

client = anthropic.Anthropic()
HAIKU = "claude-haiku-4-5"
SONNET = "claude-sonnet-4-5"
print(f"Models: worker={HAIKU}, analyst={SONNET}")

## 1. Document AI Landscape

Before we code, here's the ecosystem:

| Tool | What it does | Best for | Cost |
|------|-------------|----------|------|
| **pdfplumber** | Extract text + tables from native PDFs | Machine-generated PDFs (invoices, reports) | Free |
| **pypdf** | Page ops, metadata, text extraction | Splitting, merging, metadata reads | Free |
| **Claude PDF block** | Native PDF understanding via API | Any PDF, complex layouts | API cost |
| **Tesseract** | OCR for scanned images | True scans (no embedded text) | Free |
| **AWS Textract** | Forms, tables, signatures | Production-grade form extraction | \$\$ per page |

### Two Fundamental Approaches

```
NATIVE PDF (machine-generated)         SCANNED/VISUAL PDF
         │                                      │
    pdfplumber                           Claude PDF block
    (fast, free)                         (flexible, costs tokens)
         │                                      │
    Structured text ──────────────────── Structured text
                           │
                    Claude for
                    semantic understanding
                    + structured extraction
```

**Decision rule:**
- Native PDF with embedded text → pdfplumber first (fast, free, no API call)
- Scanned or complex layout → Claude PDF document block
- Need semantic reasoning ("what's the payment terms clause?") → always Claude

In [ ]:
# ──────────────────────────────────────────────────────────
# Create sample documents for this lesson
# (In real usage, you'd load your own PDFs)
# ──────────────────────────────────────────────────────────
from fpdf import FPDF

def create_sample_invoice(path: str = "sample_invoice.pdf"):
    """Create a realistic-looking invoice PDF for demonstration."""
    pdf = FPDF()
    pdf.add_page()

    pdf.set_font("Helvetica", "B", size=20)
    pdf.cell(0, 12, "INVOICE", ln=True, align="C")
    pdf.ln(2)

    pdf.set_font("Helvetica", size=11)
    pdf.cell(95, 7, "Acme AI Solutions Ltd.", ln=False)
    pdf.cell(95, 7, "INVOICE #: INV-2024-0042", ln=True)
    pdf.cell(95, 7, "42 Neural Net Lane, San Francisco CA 94105", ln=False)
    pdf.cell(95, 7, "DATE: 2024-11-15", ln=True)
    pdf.cell(95, 7, "contact@acme-ai.com", ln=False)
    pdf.cell(95, 7, "DUE DATE: 2024-12-15", ln=True)
    pdf.ln(5)

    pdf.set_font("Helvetica", "B", size=11)
    pdf.cell(0, 7, "BILL TO:", ln=True)
    pdf.set_font("Helvetica", size=11)
    pdf.cell(0, 7, "TechCorp International", ln=True)
    pdf.cell(0, 7, "100 Silicon Valley Blvd, Palo Alto CA 94301", ln=True)
    pdf.cell(0, 7, "accounts@techcorp.com", ln=True)
    pdf.ln(5)

    pdf.set_fill_color(220, 220, 220)
    pdf.set_font("Helvetica", "B", size=10)
    pdf.cell(90, 8, "DESCRIPTION", border=1, fill=True)
    pdf.cell(25, 8, "QTY", border=1, fill=True, align="C")
    pdf.cell(35, 8, "UNIT PRICE", border=1, fill=True, align="R")
    pdf.cell(40, 8, "TOTAL", border=1, fill=True, align="R", ln=True)

    items = [
        ("AI Agent Development (hrs)", 40, 250.00),
        ("LLM Fine-tuning Service", 1, 3500.00),
        ("GPU Cloud Credits", 100, 0.85),
        ("API Integration & Testing (hrs)", 12, 200.00),
        ("Monthly Maintenance Retainer", 1, 1200.00),
    ]

    pdf.set_font("Helvetica", size=10)
    for desc, qty, price in items:
        total = qty * price
        pdf.cell(90, 7, desc, border=1)
        pdf.cell(25, 7, str(qty), border=1, align="C")
        pdf.cell(35, 7, f"${price:,.2f}", border=1, align="R")
        pdf.cell(40, 7, f"${total:,.2f}", border=1, align="R", ln=True)

    pdf.ln(3)
    subtotal = sum(q * p for _, q, p in items)
    tax = subtotal * 0.0875
    total = subtotal + tax

    pdf.set_font("Helvetica", size=10)
    pdf.cell(150, 7, "Subtotal:", align="R")
    pdf.cell(40, 7, f"${subtotal:,.2f}", align="R", ln=True)
    pdf.cell(150, 7, "Sales Tax (8.75%):", align="R")
    pdf.cell(40, 7, f"${tax:,.2f}", align="R", ln=True)
    pdf.set_font("Helvetica", "B", size=11)
    pdf.cell(150, 8, "TOTAL DUE:", align="R")
    pdf.cell(40, 8, f"${total:,.2f}", align="R", ln=True)

    pdf.ln(8)
    pdf.set_font("Helvetica", size=9)
    pdf.cell(0, 6, "PAYMENT TERMS: Net 30. Wire transfer to: Acme AI Solutions, "
             "Routing: 021000021, Acct: 4738291056", ln=True)
    pdf.cell(0, 6, "Late payments subject to 1.5% monthly interest charge.", ln=True)

    pdf.output(path)
    print(f"✅ Created: {path}")
    return path


def create_research_abstract(path: str = "research_abstract.pdf"):
    """Create a sample research paper abstract PDF."""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_margins(20, 20, 20)

    pdf.set_font("Helvetica", "B", size=14)
    pdf.multi_cell(0, 8, "Scalable Multi-Agent Coordination via Shared Blackboard Architectures "
                   "for Enterprise Document Processing")
    pdf.ln(4)

    pdf.set_font("Helvetica", "I", size=10)
    pdf.cell(0, 6, "J. Smith, A. Kumar, L. Chen  |  Stanford AI Lab / MIT CSAIL", ln=True)
    pdf.ln(3)

    pdf.set_font("Helvetica", "B", size=11)
    pdf.cell(0, 7, "Abstract", ln=True)
    pdf.set_font("Helvetica", size=10)
    abstract_text = (
        "We present BlackboardAgent, a novel multi-agent framework for enterprise document "
        "processing that achieves 94.7% extraction accuracy on a benchmark of 10,000 mixed-format "
        "documents including PDFs, scanned forms, and contracts. Our architecture employs a shared "
        "blackboard coordination layer inspired by the HEARSAY-II speech recognition system, "
        "enabling specialist agents to collaborate without direct point-to-point communication. "
        "Compared to single-agent baselines (78.3% accuracy), our approach demonstrates a 16.4 "
        "percentage point improvement while reducing per-document API cost by 43% through "
        "intelligent task routing between OCR, vision, and text-extraction specialists. "
        "We evaluate on three enterprise benchmarks: InvoiceNet-2K (invoices), ContractDB-500 "
        "(legal contracts), and FormParse-1K (government forms). Our open-source implementation "
        "is available at github.com/example/blackboard-agent."
    )
    pdf.multi_cell(0, 6, abstract_text)
    pdf.ln(4)

    pdf.set_font("Helvetica", "B", size=11)
    pdf.cell(0, 7, "Key Results", ln=True)
    pdf.set_font("Helvetica", size=10)
    pdf.set_fill_color(230, 230, 230)

    headers = ["Benchmark", "Single Agent", "BlackboardAgent", "Delta Accuracy"]
    widths = [55, 35, 45, 35]
    for h, w in zip(headers, widths):
        pdf.cell(w, 7, h, border=1, fill=True)
    pdf.ln()

    rows = [
        ("InvoiceNet-2K", "81.2%", "95.8%", "+14.6pp"),
        ("ContractDB-500", "74.1%", "93.2%", "+19.1pp"),
        ("FormParse-1K", "79.6%", "95.1%", "+15.5pp"),
        ("Overall", "78.3%", "94.7%", "+16.4pp"),
    ]
    for row in rows:
        for val, w in zip(row, widths):
            pdf.cell(w, 7, val, border=1)
        pdf.ln()

    pdf.ln(4)
    pdf.set_font("Helvetica", "B", size=10)
    pdf.cell(0, 6, "Keywords: ", ln=False)
    pdf.set_font("Helvetica", "I", size=10)
    pdf.cell(0, 6, "multi-agent systems, document AI, PDF extraction, blackboard architecture, "
             "enterprise automation", ln=True)

    pdf.output(path)
    print(f"✅ Created: {path}")
    return path


invoice_path = create_sample_invoice()
paper_path = create_research_abstract()
print("\nDocuments ready for analysis!")

## 2. Approach 1 — pdfplumber: Fast Native PDF Extraction

`pdfplumber` wraps `pdfminer.six` and adds a clean API for text + table extraction from **machine-generated** PDFs (where text is embedded, not scanned).

**When to use it:**
- PDF was generated by software (Word → PDF, LaTeX, browser print)
- You need tables with proper cell boundaries
- Cost matters (zero API tokens)

**When NOT to use it:**
- Scanned documents (no embedded text → returns empty string)
- Complex multi-column layouts that confuse text order
- You need to *understand* the content semantically

In [ ]:
import pdfplumber
import pandas as pd

def extract_with_pdfplumber(pdf_path: str) -> dict:
    """
    Extract text and tables from a native PDF.
    Returns dict with 'pages' (text per page) and 'tables' (list of DataFrames).
    """
    result = {"pages": [], "tables": [], "metadata": {}}

    with pdfplumber.open(pdf_path) as pdf:
        result["metadata"] = {
            "num_pages": len(pdf.pages),
            "creator": pdf.metadata.get("Creator", "Unknown"),
        }

        for page_num, page in enumerate(pdf.pages, 1):
            text = page.extract_text() or ""
            result["pages"].append({
                "page": page_num,
                "text": text,
                "char_count": len(text),
            })

            raw_tables = page.extract_tables()
            for t_idx, table in enumerate(raw_tables):
                if table and len(table) > 1:
                    df = pd.DataFrame(table[1:], columns=table[0])
                    if df.dropna(how="all").shape[0] > 0:
                        result["tables"].append({
                            "page": page_num,
                            "table_index": t_idx,
                            "dataframe": df,
                            "rows": len(df),
                            "cols": len(df.columns),
                        })

    return result


# ── Extract from our invoice ──────────────────────────────────────────
print("=" * 55)
print("PDFPLUMBER EXTRACTION — INVOICE")
print("=" * 55)
invoice_data = extract_with_pdfplumber(invoice_path)

print(f"\n📄 Pages: {invoice_data['metadata']['num_pages']}")
print(f"📊 Tables found: {len(invoice_data['tables'])}")
print("\n--- Raw text (first 600 chars) ---")
print(invoice_data["pages"][0]["text"][:600])

if invoice_data["tables"]:
    t = invoice_data["tables"][0]
    print(f"\n--- Table 0 ({t['rows']} rows x {t['cols']} cols) ---")
    print(t["dataframe"].to_string(index=False))

# 💡 EXPERIMENT: Try extract_with_pdfplumber(paper_path) to compare results

## 3. Approach 2 — Claude's Native PDF Understanding

Since late 2024, Claude supports **native PDF input** via the `document` content block. You send the entire PDF as base64 and Claude reads it with full layout understanding — including multi-column text, embedded images, and complex tables.

**Advantages over pdfplumber:**
- Works on any PDF (including scanned ones with embedded images)
- Understands *layout* and *semantics*, not just raw characters
- Can answer questions, summarize, and extract in one call
- No separate OCR step needed

**Cost consideration:** A typical 1-page PDF costs ~800–2000 input tokens.

### The Document Block Format

```python
{
    "type": "document",
    "source": {
        "type": "base64",
        "media_type": "application/pdf",
        "data": "<base64_encoded_pdf>"
    }
}
```

This goes in the `content` array of a user message alongside a `text` block with your question.

In [ ]:
def load_pdf_as_b64(path: str) -> str:
    """Load a PDF file and encode as base64 for the Claude API."""
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")


def claude_read_pdf(
    pdf_path: str,
    question: str,
    model: str = HAIKU,
    max_tokens: int = 1024,
) -> str:
    """
    Send a PDF to Claude and ask a question about it.
    Uses the native document content block — no text pre-extraction needed.
    """
    pdf_b64 = load_pdf_as_b64(pdf_path)

    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": pdf_b64,
                        },
                    },
                    {"type": "text", "text": question},
                ],
            }
        ],
    )
    return response.content[0].text


# Test — summarize the invoice
print("=" * 55)
print("CLAUDE PDF — INVOICE SUMMARY")
print("=" * 55)
summary = claude_read_pdf(
    invoice_path,
    "Summarize this invoice in 2-3 sentences: who is billing whom, for what, "
    "and what is the total amount due?",
)
print(summary)

print("\n" + "=" * 55)
print("CLAUDE PDF — RESEARCH PAPER FINDINGS")
print("=" * 55)
findings = claude_read_pdf(
    paper_path,
    "What are the three benchmarks tested in this paper, and what accuracy "
    "did BlackboardAgent achieve on each?",
    model=SONNET,  # 💡 EXPERIMENT: swap to HAIKU and compare accuracy
)
print(findings)

## 4. Structured Extraction — Documents → Pydantic Models

Free-text summaries are useful, but **agents need structured data**. The pattern:

```
PDF → Claude + tool_use → Pydantic model → downstream action
```

We force Claude to fill a schema by defining an extraction tool with `tool_choice={"type": "tool", "name": "..."}`. This is the same pattern from L10 (Structured Outputs) applied to documents.

**Why tool_use over asking for JSON:**
- Schema is validated by Pydantic before it reaches your code
- Claude can't skip fields or change types
- You get a parsed Python object, not a string to re-parse

In [ ]:
from pydantic import BaseModel, Field

class LineItem(BaseModel):
    description: str
    quantity: float
    unit_price: float
    total: float

class InvoiceExtract(BaseModel):
    invoice_number: str
    invoice_date: str
    due_date: str
    vendor_name: str
    vendor_email: Optional[str] = None
    client_name: str
    client_email: Optional[str] = None
    line_items: List[LineItem]
    subtotal: float
    tax_rate_pct: Optional[float] = None
    tax_amount: float
    total_due: float
    payment_terms: Optional[str] = None
    bank_routing: Optional[str] = None
    bank_account: Optional[str] = None


EXTRACT_INVOICE_TOOL = {
    "name": "extract_invoice",
    "description": "Extract structured invoice data from a document",
    "input_schema": InvoiceExtract.model_json_schema(),
}


def extract_invoice_structured(pdf_path: str) -> InvoiceExtract:
    """
    Extract structured invoice data from a PDF using Claude + Pydantic.
    Returns a validated InvoiceExtract object.
    """
    pdf_b64 = load_pdf_as_b64(pdf_path)

    response = client.messages.create(
        model=SONNET,
        max_tokens=2048,
        tools=[EXTRACT_INVOICE_TOOL],
        tool_choice={"type": "tool", "name": "extract_invoice"},
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_b64,
                    },
                },
                {
                    "type": "text",
                    "text": "Extract all invoice fields into the extract_invoice tool. "
                           "For monetary values, provide numbers only (no $ sign). "
                           "For tax_rate_pct, provide the percentage as a number (e.g. 8.75).",
                },
            ],
        }],
    )

    tool_use = next(b for b in response.content if b.type == "tool_use")
    return InvoiceExtract(**tool_use.input)


print("=" * 55)
print("STRUCTURED INVOICE EXTRACTION")
print("=" * 55)
invoice = extract_invoice_structured(invoice_path)

print(f"\n📋 Invoice: {invoice.invoice_number}")
print(f"📅 Date: {invoice.invoice_date} | Due: {invoice.due_date}")
print(f"🏢 Vendor: {invoice.vendor_name}")
print(f"👤 Client: {invoice.client_name}")
print(f"\n📦 Line Items ({len(invoice.line_items)}):")
for item in invoice.line_items:
    print(f"  • {item.description}: {item.quantity} x ${item.unit_price:,.2f} = ${item.total:,.2f}")
print(f"\n💰 Subtotal: ${invoice.subtotal:,.2f}")
print(f"   Tax ({invoice.tax_rate_pct}%): ${invoice.tax_amount:,.2f}")
print(f"   TOTAL DUE: ${invoice.total_due:,.2f}")
if invoice.payment_terms:
    print(f"\n📝 Terms: {invoice.payment_terms}")

# 💡 EXPERIMENT: Add a field to InvoiceExtract (e.g., 'currency: str = "USD"')
# and re-run — Claude automatically fills the new field.

## 5. Table Extraction — pdfplumber vs Claude Hybrid

Tables are the trickiest part of document AI. You have two strategies:

| Strategy | Accuracy | Cost | Best for |
|----------|----------|------|----------|
| **pdfplumber** `extract_tables()` | High (native PDFs) | Free | Invoices, reports with clear grid lines |
| **Claude PDF** | Very high (any table) | API tokens | Irregular tables, merged cells, scanned |

**The hybrid pattern:** Use pdfplumber first; if tables are malformed or missing, fall back to Claude.

In [ ]:
from tabulate import tabulate

def extract_tables_hybrid(
    pdf_path: str,
    fallback_to_claude: bool = True,
) -> List[pd.DataFrame]:
    """Extract tables with pdfplumber, falling back to Claude if needed."""
    tables = []

    # Strategy 1: pdfplumber (free)
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for raw in (page.extract_tables() or []):
                if raw and len(raw) > 1:
                    df = pd.DataFrame(raw[1:], columns=raw[0])
                    if df.dropna(how="all").shape[0] > 0:
                        tables.append(df)

    if tables:
        print(f"✅ pdfplumber found {len(tables)} table(s)")
        return tables

    # Strategy 2: Claude fallback
    if fallback_to_claude:
        print("⚠️  pdfplumber found no tables → falling back to Claude")
        pdf_b64 = load_pdf_as_b64(pdf_path)

        response = client.messages.create(
            model=HAIKU,
            max_tokens=2048,
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64},
                    },
                    {
                        "type": "text",
                        "text": "Extract all tables from this document. Return JSON: a list of "
                               "objects with 'headers' (list of column names) and 'rows' "
                               "(list of lists of string values). Return only JSON, no explanation.",
                    },
                ],
            }],
        )

        raw = response.content[0].text.strip()
        if raw.startswith("```"):
            raw = re.sub(r"```(?:json)?\n?", "", raw).rstrip("`").strip()

        try:
            parsed = json.loads(raw)
            for t in (parsed if isinstance(parsed, list) else []):
                df = pd.DataFrame(t.get("rows", []), columns=t.get("headers", []))
                tables.append(df)
            print(f"✅ Claude extracted {len(tables)} table(s)")
        except json.JSONDecodeError:
            print("❌ Could not parse Claude's table output")

    return tables


print("=" * 55)
print("TABLE EXTRACTION — RESEARCH PAPER")
print("=" * 55)
tables = extract_tables_hybrid(paper_path)
for i, df in enumerate(tables):
    print(f"\n📊 Table {i+1} ({df.shape[0]} rows x {df.shape[1]} cols):")
    print(tabulate(df, headers="keys", tablefmt="grid", showindex=False))

print("\n" + "=" * 55)
print("TABLE EXTRACTION — INVOICE")
print("=" * 55)
tables = extract_tables_hybrid(invoice_path)
for i, df in enumerate(tables):
    print(f"\n📊 Table {i+1}:")
    print(tabulate(df.head(8), headers="keys", tablefmt="grid", showindex=False))

## 6. Document Tools for Agent Loops

Now we wire document capabilities into an **agent tool schema** so Claude can call them in a ReAct loop. The agent decides *which* document operation to run, just like it would call a web search or calculator.

### Document Tool Suite

```
DOCUMENT_TOOLS
├── read_pdf_text        → extract raw text from PDF
├── extract_tables       → get tables as structured JSON
├── extract_invoice_fields → fill InvoiceExtract schema
└── summarize_document   → generate concise summary
```

The agent orchestrates these based on what the user asks. Same architecture as L3's tool loop — documents are just another tool capability.

In [ ]:
DOCUMENT_TOOLS = [
    {
        "name": "read_pdf_text",
        "description": "Extract the raw text content from a PDF document. "
                       "Use this to read or search the full document text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pdf_path": {
                    "type": "string",
                    "description": "Path to the PDF file on disk",
                },
                "page_number": {
                    "type": "integer",
                    "description": "Optional: extract from a specific page only (1-indexed). Omit for all pages.",
                },
            },
            "required": ["pdf_path"],
        },
    },
    {
        "name": "extract_tables",
        "description": "Extract all tables from a PDF as structured JSON data.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pdf_path": {"type": "string", "description": "Path to the PDF file"},
            },
            "required": ["pdf_path"],
        },
    },
    {
        "name": "extract_invoice_fields",
        "description": "Extract structured fields from an invoice PDF: vendor, client, "
                       "line items, totals, payment terms. Use for invoice documents.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pdf_path": {"type": "string", "description": "Path to the invoice PDF"},
            },
            "required": ["pdf_path"],
        },
    },
    {
        "name": "summarize_document",
        "description": "Generate a concise summary of a PDF. Good for quick understanding "
                       "before deeper extraction.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pdf_path": {"type": "string", "description": "Path to the PDF file"},
                "focus": {
                    "type": "string",
                    "description": "Optional: aspect to focus on (e.g. 'financials', 'key findings')",
                },
            },
            "required": ["pdf_path"],
        },
    },
]


@dataclass
class ToolResult:
    tool_name: str
    success: bool
    output: Any
    error: Optional[str] = None


def execute_document_tool(tool_name: str, tool_input: dict) -> ToolResult:
    """Execute a document tool call and return JSON-serializable data."""
    pdf_path = tool_input.get("pdf_path", "")

    if not Path(pdf_path).exists():
        return ToolResult(tool_name, False, None, f"File not found: {pdf_path}")

    try:
        if tool_name == "read_pdf_text":
            with pdfplumber.open(pdf_path) as pdf:
                page_num = tool_input.get("page_number")
                pages = [pdf.pages[page_num - 1]] if page_num else pdf.pages
                text_parts = [
                    {"page": i + 1, "text": page.extract_text() or ""}
                    for i, page in enumerate(pages)
                ]
                return ToolResult(tool_name, True, {"pages": text_parts, "total_pages": len(pdf.pages)})

        elif tool_name == "extract_tables":
            tables_data = []
            with pdfplumber.open(pdf_path) as pdf:
                for pn, page in enumerate(pdf.pages, 1):
                    for raw in (page.extract_tables() or []):
                        if raw and len(raw) > 1:
                            headers = [str(h) if h else f"Col{i}" for i, h in enumerate(raw[0])]
                            rows = [[str(c) if c else "" for c in row] for row in raw[1:]]
                            tables_data.append({"page": pn, "headers": headers, "rows": rows})
            return ToolResult(tool_name, True, {"tables": tables_data, "count": len(tables_data)})

        elif tool_name == "extract_invoice_fields":
            inv = extract_invoice_structured(pdf_path)
            return ToolResult(tool_name, True, inv.model_dump())

        elif tool_name == "summarize_document":
            focus = tool_input.get("focus", "")
            q = "Summarize this document concisely."
            if focus:
                q += f" Focus on: {focus}."
            summary = claude_read_pdf(pdf_path, q, model=HAIKU)
            return ToolResult(tool_name, True, {"summary": summary})

        else:
            return ToolResult(tool_name, False, None, f"Unknown tool: {tool_name}")

    except Exception as e:
        return ToolResult(tool_name, False, None, str(e))


print("✅ DOCUMENT_TOOLS and execute_document_tool() defined")
print(f"   Tools: {[t['name'] for t in DOCUMENT_TOOLS]}")

In [ ]:
@dataclass
class AgentState:
    task: str
    pdf_paths: List[str]
    messages: List[dict] = field(default_factory=list)
    tool_results: List[ToolResult] = field(default_factory=list)
    total_input_tokens: int = 0
    total_output_tokens: int = 0


def run_document_agent(
    task: str,
    pdf_paths: List[str],
    model: str = SONNET,
    max_rounds: int = 6,
    verbose: bool = True,
) -> str:
    """
    Run a document-aware ReAct agent.
    The agent selects which document tools to call based on the task.
    """
    state = AgentState(task=task, pdf_paths=pdf_paths)

    system = (
        "You are a Document AI Agent with access to tools that extract text, tables, "
        "and structured fields from PDF files. Use tools to answer the user's request accurately. "
        "When you have enough information, provide a clear structured answer.\n\n"
        f"Available documents: {', '.join(pdf_paths)}"
    )

    state.messages.append({"role": "user", "content": task})

    for round_num in range(1, max_rounds + 1):
        if verbose:
            print(f"\n{'─'*45}")
            print(f"Round {round_num}")
            print(f"{'─'*45}")

        response = client.messages.create(
            model=model,
            max_tokens=2048,
            system=system,
            tools=DOCUMENT_TOOLS,
            messages=state.messages,
        )

        state.total_input_tokens += response.usage.input_tokens
        state.total_output_tokens += response.usage.output_tokens

        assistant_content = []
        tool_calls = []

        for block in response.content:
            if block.type == "text":
                if verbose and block.text.strip():
                    print(f"🤖 Agent: {block.text.strip()[:300]}")
                assistant_content.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                tool_calls.append(block)
                if verbose:
                    print(f"🔧 Tool: {block.name}({json.dumps(block.input)})")
                assistant_content.append({
                    "type": "tool_use",
                    "id": block.id,
                    "name": block.name,
                    "input": block.input,
                })

        state.messages.append({"role": "assistant", "content": assistant_content})

        if response.stop_reason == "end_turn" and not tool_calls:
            final_text = next(
                (b["text"] for b in assistant_content if b["type"] == "text"), ""
            )
            cost_usd = (
                state.total_input_tokens * 3 + state.total_output_tokens * 15
            ) / 1_000_000
            if verbose:
                print(f"\n✅ Done in {round_num} round(s) | "
                      f"{state.total_input_tokens}↑ {state.total_output_tokens}↓ tokens | "
                      f"~${cost_usd:.4f}")
            return final_text

        # Execute tools
        tool_result_content = []
        for tc in tool_calls:
            result = execute_document_tool(tc.name, tc.input)
            state.tool_results.append(result)
            result_text = (
                json.dumps(result.output, indent=2)[:3000]
                if result.success
                else f"ERROR: {result.error}"
            )
            if verbose:
                status = "✅" if result.success else "❌"
                print(f"   {status} {tc.name} → {result_text[:120]}...")
            tool_result_content.append({
                "type": "tool_result",
                "tool_use_id": tc.id,
                "content": result_text,
            })

        state.messages.append({"role": "user", "content": tool_result_content})

    return "Agent did not finish within the allowed rounds."


print("✅ run_document_agent() defined")

In [ ]:
print("=" * 55)
print("DOCUMENT AGENT DEMO — INVOICE ANALYSIS")
print("=" * 55)

answer = run_document_agent(
    task=(
        f"Analyze the invoice at {invoice_path}. "
        "Tell me: (1) who is billing whom, (2) the largest line item by total cost, "
        "(3) when payment is due, and (4) the wire transfer details."
    ),
    pdf_paths=[invoice_path],
)

print("\n" + "=" * 55)
print("FINAL ANSWER")
print("=" * 55)
print(answer)

# 💡 EXPERIMENT: Try the agent on the research paper:
# run_document_agent(
#     task=f"From {paper_path}, extract the main contribution and best accuracy achieved.",
#     pdf_paths=[paper_path],
# )

## 7. Multi-Page Documents & Chunking

Real-world documents are long — 50-page contracts, 200-page annual reports. Sending a 200-page PDF in one call would:
1. Hit `max_tokens` limits (especially on complex PDFs)
2. Cost \$10–20 per call for large models
3. Cause Claude to lose focus on early pages ("lost in the middle" problem)

### Chunking Strategies

| Strategy | When | How |
|----------|------|-----|
| **Page-by-page** | Uniform docs (form collections) | Process each page independently, combine |
| **Section-based** | Structured docs (contracts, reports) | Detect headers → split by section |
| **Sliding window** | Dense technical docs | Overlapping chunks, dedup outputs |
| **Map-reduce** | Long docs needing synthesis | Extract per chunk → merge summaries |

We'll implement **map-reduce** — the same pattern from L35 applied to documents.

In [ ]:
def chunk_pdf_by_pages(pdf_path: str, chunk_size: int = 3) -> List[dict]:
    """Split a PDF into chunks of `chunk_size` pages each."""
    chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        pages = pdf.pages
        for start in range(0, len(pages), chunk_size):
            end = min(start + chunk_size, len(pages))
            chunk_text = "".join(
                (page.extract_text() or "") + "\n"
                for page in pages[start:end]
            ).strip()
            if chunk_text:
                chunks.append({
                    "start_page": start + 1,
                    "end_page": end,
                    "text": chunk_text,
                    "chunk_id": f"pages_{start+1}_{end}",
                })
    return chunks


def map_reduce_document(
    pdf_path: str,
    extraction_task: str,
    chunk_size: int = 3,
    model: str = HAIKU,
) -> str:
    """
    Map-reduce over a PDF:
    - MAP:    Extract info from each chunk independently (parallelizable)
    - REDUCE: Merge chunk extractions into a final de-duplicated answer
    """
    chunks = chunk_pdf_by_pages(pdf_path, chunk_size)
    print(f"📄 Split into {len(chunks)} chunk(s) of up to {chunk_size} page(s)")

    # ── MAP phase ────────────────────────────────────────────────────────
    chunk_results = []
    for chunk in chunks:
        prompt = (
            f"Document chunk (pages {chunk['start_page']}–{chunk['end_page']}):\n\n"
            f"{chunk['text']}\n\n"
            f"Task: {extraction_task}\n"
            "Extract relevant information from THIS chunk only. "
            "If no relevant info, reply: NO_RELEVANT_INFO"
        )
        response = client.messages.create(
            model=model,
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}],
        )
        result = response.content[0].text.strip()
        found = "NO_RELEVANT_INFO" not in result
        if found:
            chunk_results.append(f"[Pages {chunk['start_page']}–{chunk['end_page']}]\n{result}")
        print(f"  {chunk['chunk_id']}: {'✅ found info' if found else '⬜ no info'}")

    if not chunk_results:
        return "No relevant information found in the document."

    # ── REDUCE phase ──────────────────────────────────────────────────────
    combined = "\n\n".join(chunk_results)
    reduce_prompt = (
        f"You extracted the following from different parts of a document:\n\n{combined}\n\n"
        f"Original task: {extraction_task}\n\n"
        "Synthesize into a single, coherent, de-duplicated answer."
    )
    final = client.messages.create(
        model=SONNET,
        max_tokens=1024,
        messages=[{"role": "user", "content": reduce_prompt}],
    )
    return final.content[0].text.strip()


print("=" * 55)
print("MAP-REDUCE — RESEARCH PAPER")
print("=" * 55)
result = map_reduce_document(
    paper_path,
    "What quantitative results does this paper report? Extract all percentages and what they mean.",
    chunk_size=1,  # 1 page per chunk (paper is short)
)
print(f"\n📊 Synthesized Result:\n{result}")

# 💡 EXPERIMENT: Try map_reduce_document() on a longer PDF from arxiv.org

## 8. Research Paper Assistant — A Real Use Case

Putting it all together: an agent that extracts structured metadata from research papers AND generates a practitioner-friendly summary. This is directly usable for AI literature review.

The agent will:
1. Extract structured metadata (title, authors, benchmarks, accuracy) via `tool_use`
2. Generate a 3-sentence "worth reading?" summary via plain completion
3. Return both in a single function call

In [ ]:
class PaperMetadata(BaseModel):
    title: str
    authors: List[str]
    institutions: List[str]
    main_contribution: str
    benchmarks: List[str] = Field(default_factory=list)
    best_accuracy_reported: Optional[str] = None
    baseline_compared: Optional[str] = None
    open_source_url: Optional[str] = None
    keywords: List[str] = Field(default_factory=list)


EXTRACT_PAPER_TOOL = {
    "name": "extract_paper_metadata",
    "description": "Extract structured metadata from a research paper PDF",
    "input_schema": PaperMetadata.model_json_schema(),
}


def analyze_research_paper(pdf_path: str) -> Tuple[PaperMetadata, str]:
    """Extract structured metadata + practitioner summary from a research paper."""
    pdf_b64 = load_pdf_as_b64(pdf_path)
    doc_block = {
        "type": "document",
        "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64},
    }

    # Step 1: Structured metadata extraction
    meta_response = client.messages.create(
        model=SONNET,
        max_tokens=2048,
        tools=[EXTRACT_PAPER_TOOL],
        tool_choice={"type": "tool", "name": "extract_paper_metadata"},
        messages=[{"role": "user", "content": [
            doc_block,
            {"type": "text", "text": "Extract metadata from this research paper."}
        ]}],
    )
    tool_use = next(b for b in meta_response.content if b.type == "tool_use")
    metadata = PaperMetadata(**tool_use.input)

    # Step 2: Practitioner summary
    summary_response = client.messages.create(
        model=HAIKU,
        max_tokens=512,
        messages=[{"role": "user", "content": [
            doc_block,
            {
                "type": "text",
                "text": "Write a 3-sentence summary for an AI practitioner deciding if this paper is worth reading. "
                       "Cover: what problem it solves, how it solves it, and the practical takeaway."
            }
        ]}],
    )
    summary = summary_response.content[0].text.strip()

    return metadata, summary


print("=" * 55)
print("RESEARCH PAPER ASSISTANT")
print("=" * 55)
metadata, summary = analyze_research_paper(paper_path)

print(f"\n📄 {metadata.title}")
print(f"👥 {', '.join(metadata.authors)}")
print(f"🏛  {', '.join(metadata.institutions)}")
print(f"\n🎯 Contribution:\n{metadata.main_contribution}")
print(f"\n📊 Benchmarks: {', '.join(metadata.benchmarks)}")
print(f"📈 Best accuracy: {metadata.best_accuracy_reported}")
if metadata.open_source_url:
    print(f"💻 Code: {metadata.open_source_url}")
print(f"\n🏷  Keywords: {', '.join(metadata.keywords)}")
print(f"\n📝 Practitioner Summary:\n{summary}")

# 💡 EXPERIMENT: Download a real ML paper from arxiv.org and run this on it!

## 9. Choosing Your Document AI Strategy

| Situation | Recommended Approach | Why |
|-----------|---------------------|-----|
| Native PDF, need text | pdfplumber | Free, fast, zero API cost |
| Native PDF, need tables | pdfplumber `extract_tables()` | Accurate cell geometry |
| Scanned / complex layout | Claude PDF document block | Vision beats text extraction |
| Need structured fields | Claude + `tool_use` + Pydantic | Validated, typed output |
| Long doc (>10 pages) | Map-reduce chunking | Avoid context overflow + cost |
| QA over a document | DocumentAgent (ReAct) | Dynamic tool selection |
| Research paper analysis | `analyze_research_paper()` | Structured + narrative |

### Cost Reference (per document, approximate)

| Approach | ~Tokens | ~Cost (Haiku) | ~Cost (Sonnet) |
|----------|---------|---------------|----------------|
| pdfplumber only | 0 | \$0.000 | \$0.000 |
| Claude PDF (1-page) | ~1,500 | \$0.001 | \$0.005 |
| Structured extraction (1-page) | ~2,000 | \$0.001 | \$0.007 |
| DocumentAgent (3-5 tool calls) | ~6,000 | \$0.004 | \$0.020 |
| Map-reduce (10 pages) | ~8,000 | \$0.006 | \$0.025 |

**Rule of thumb:** pdfplumber for structure-first extraction; add Claude only when you need semantic understanding.

In [ ]:
@dataclass
class ProcessingStats:
    documents_processed: int = 0
    total_input_tokens: int = 0
    total_output_tokens: int = 0
    total_cost_usd: float = 0.0
    errors: List[str] = field(default_factory=list)

    def add_usage(self, input_tokens: int, output_tokens: int, model: str):
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        rate_in = 0.80 if "haiku" in model else 3.00
        rate_out = 4.00 if "haiku" in model else 15.00
        self.total_cost_usd += (input_tokens * rate_in + output_tokens * rate_out) / 1_000_000

    def report(self):
        print(f"\n📊 Processing Stats:")
        print(f"   Documents: {self.documents_processed}")
        print(f"   Tokens: {self.total_input_tokens:,}↑ {self.total_output_tokens:,}↓")
        print(f"   Estimated cost: ${self.total_cost_usd:.4f}")
        if self.errors:
            print(f"   Errors ({len(self.errors)}):")
            for e in self.errors[:3]:
                print(f"     • {e}")


class SafeDocumentProcessor:
    """
    Production-safe document processor with:
    - Per-session budget cap
    - Hybrid extraction (pdfplumber → Claude fallback)
    - Error handling with graceful degradation
    - Cost tracking
    """

    def __init__(self, max_cost_usd: float = 1.00, model: str = HAIKU):
        self.max_cost_usd = max_cost_usd
        self.model = model
        self.stats = ProcessingStats()

    def _check_budget(self) -> bool:
        if self.stats.total_cost_usd >= self.max_cost_usd:
            print(f"⚠️  Budget cap reached (${self.stats.total_cost_usd:.4f} >= ${self.max_cost_usd:.2f})")
            return False
        return True

    def extract_text(self, pdf_path: str) -> Optional[str]:
        """Extract text: pdfplumber first, Claude fallback."""
        if not self._check_budget():
            return None
        try:
            with pdfplumber.open(pdf_path) as pdf:
                text = "\n\n".join(p.extract_text() or "" for p in pdf.pages).strip()
            if text:
                self.stats.documents_processed += 1
                return text
        except Exception as e:
            self.stats.errors.append(f"pdfplumber: {e}")

        # Claude fallback
        try:
            text = claude_read_pdf(pdf_path, "Extract all text preserving structure.", model=self.model)
            self.stats.add_usage(2000, len(text) // 4, self.model)
            self.stats.documents_processed += 1
            return text
        except Exception as e:
            self.stats.errors.append(f"Claude: {e}")
            return None

    def extract_structured(self, pdf_path: str, schema_tool: dict, schema_class) -> Optional[Any]:
        """Extract structured fields using Claude + Pydantic."""
        if not self._check_budget():
            return None
        try:
            pdf_b64 = load_pdf_as_b64(pdf_path)
            response = client.messages.create(
                model=self.model,
                max_tokens=2048,
                tools=[schema_tool],
                tool_choice={"type": "tool", "name": schema_tool["name"]},
                messages=[{"role": "user", "content": [
                    {"type": "document", "source": {
                        "type": "base64", "media_type": "application/pdf", "data": pdf_b64
                    }},
                    {"type": "text", "text": f"Extract structured data using the {schema_tool['name']} tool."}
                ]}],
            )
            self.stats.add_usage(response.usage.input_tokens, response.usage.output_tokens, self.model)
            tool_use = next(b for b in response.content if b.type == "tool_use")
            self.stats.documents_processed += 1
            return schema_class(**tool_use.input)
        except Exception as e:
            self.stats.errors.append(f"Structured extraction: {e}")
            return None


# ── Demo ───────────────────────────────────────────────────────────────
print("=" * 55)
print("SAFE DOCUMENT PROCESSOR DEMO")
print("=" * 55)
processor = SafeDocumentProcessor(max_cost_usd=0.50, model=HAIKU)

text1 = processor.extract_text(invoice_path)
print(f"✅ Invoice text: {len(text1 or '')} chars")

text2 = processor.extract_text(paper_path)
print(f"✅ Paper text: {len(text2 or '')} chars")

processor.stats.report()

## 10. Ten Document AI Pitfalls

| # | Pitfall | Impact | Fix |
|---|---------|--------|-----|
| 1 | **Assuming pdfplumber works on scanned PDFs** | Empty text output | Check `extract_text()` length; fall back to Claude if empty |
| 2 | **Sending 50-page PDF as one API call** | Context overflow + \$10+ cost | Chunk by pages; use map-reduce |
| 3 | **Trusting raw table rows from pdfplumber blindly** | Merged-cell columns misaligned | Validate with `df.dropna(how='all')` + shape checks |
| 4 | **No budget cap on batch document processing** | Runaway costs on large file sets | Use `SafeDocumentProcessor(max_cost_usd=...)` |
| 5 | **Storing PDF base64 in message history** | Context blows up on multi-turn | Re-encode per call; never keep raw bytes in messages list |
| 6 | **Using Sonnet for every extraction** | 5× cost vs Haiku for simple tasks | Haiku for text/table extraction; Sonnet for reasoning |
| 7 | **Not validating Pydantic output against business rules** | Silent wrong totals | Add validators: e.g. `sum(line_items) == subtotal` |
| 8 | **Single-page chunking on equation-heavy papers** | Equations lose context | Overlap chunks by 1 page; use section-aware splitting |
| 9 | **No extraction accuracy eval** | You don't know if it actually works | Build golden set: 10 hand-labeled docs, measure field accuracy |
| 10 | **Ignoring PDF encoding/encryption** | pdfplumber throws silently | Check `pdf.metadata` for encryption flags; handle with pypdf `decrypt()` |

In [ ]:
# ── Pitfall #5 illustrated: DON'T keep base64 in conversation history ──
#
# WRONG — bloats context every turn (PDF re-sent each round):
# messages = [{"role": "user", "content": [
#     {"type": "document", "source": {"type": "base64", "data": pdf_b64}},  # 50KB+
#     {"type": "text", "text": "Follow-up question..."},
# ]}]

# RIGHT — re-encode per question; history stays lean
def multi_turn_doc_qa(pdf_path: str, questions: List[str]) -> List[str]:
    """Multi-turn Q&A over a document without storing PDF bytes in history."""
    return [
        claude_read_pdf(pdf_path, q, model=HAIKU)
        for q in questions
    ]


questions = [
    "What is the invoice number?",
    "Which line item has the highest total cost?",
    "What is the wire transfer routing number?",
]
print("Multi-turn Q&A (efficient — no base64 accumulation):")
for q, a in zip(questions, multi_turn_doc_qa(invoice_path, questions)):
    print(f"\nQ: {q}")
    print(f"A: {a.strip()}")

## 11. Homework — 5 Tasks

1. **Real invoice pipeline:** Download 3 real invoice PDFs (or use your own). Run `extract_invoice_structured()` on each and compare accuracy. Where does it fail — missing fields? Wrong numbers?

2. **Contract clause extractor:** Create a `ContractExtract` Pydantic model with fields: `parties`, `effective_date`, `termination_clause`, `governing_law`, `payment_terms`. Run it on any contract PDF you have.

3. **Batch processor with cost dashboard:** Process 10 PDFs with `SafeDocumentProcessor`. Print a per-document breakdown and total. Add a `dry_run=True` mode that estimates cost without calling the API (based on file size).

4. **Semantic search over PDFs:** Extract text from 5 PDFs → chunk → embed with the Anthropic embeddings API → store in ChromaDB (from L7) → build a Q&A interface. This closes the RAG-for-documents loop.

5. **Accuracy golden set:** Create 5 test invoices with known ground truth (you know the correct values). Run `extract_invoice_structured()` and compute field-level accuracy (correct_fields / total_fields). This is your document AI eval harness.

---

## Track 4 Progress

| Lesson | Topic | Status |
|--------|-------|--------|
| L42 | Voice Agent Pipelines | ✅ |
| L43 | Image Generation Tools | ✅ |
| **L44** | **Document AI** | **✅ Today** |
| L45 | Track 4 Capstone | ⏳ Next |

**Next lesson (L45):** The Track 4 Capstone — a **Multimodal Agent** that combines voice input, image generation, and document understanding in a single agent loop. You'll wire together the ASR pipeline (L42), image gen tools (L43), and document AI (L44) into one coherent agent that can hear, see, read, and respond. This is the culmination of Track 4!